# Second-Order Texture Feature Analysis: Per Tumor Subregion (WT, TC, ET)

Extends `GLCM_Generate_Charts.ipynb` (and siblings) by analyzing all five texture families
across **all three tumor subregions** (WT, TC, ET) rather than WT only.

For each feature × MRI modality combination, good vs poor segmentations are compared
with 3 groups: Whole Tumor (WT), Tumor Core (TC), Enhancing Tumor (ET).

**Texture families covered:** GLCM, GLDM, GLRLM, GLSZM, NGTDM (all at scale 10 / full)

**Outputs:**
- `../../Results/final_figures/texture_subregion_glcm.pdf`
- `../../Results/final_figures/texture_subregion_gldm.pdf`
- `../../Results/final_figures/texture_subregion_glrlm.pdf`
- `../../Results/final_figures/texture_subregion_glszm.pdf`
- `../../Results/final_figures/texture_subregion_ngtdm.pdf`
- `../../Results/Json_summary/summary_texture_subregion.json`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import mannwhitneyu
import json
import pickle as pkl
import warnings

warnings.filterwarnings('ignore')

/Users/suvodeepmajumder/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/suvodeepmajumder/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Helper Functions — Data Loading

In [2]:
def load_unet_result(path):
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0] for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis=1, inplace=True)
    return df


def load_nnunet_result(path):
    with open(path) as f:
        data = json.load(f)
    WT, TC, ET, names = [], [], [], []
    for case in data['metric_per_case']:
        WT.append(case['metrics']['(2, 1, 3)']['Dice'])
        TC.append(case['metrics']['(2, 3)']['Dice'])
        ET.append(case['metrics']['(3,)']['Dice'])
        names.append(case['reference_file'].split('/')[-1].split('.')[0])
    return pd.DataFrame(zip(WT, TC, ET), columns=['WT dice', 'TC dice', 'ET dice'], index=names)


def load_TransBTS_result(path):
    with open(path) as f:
        data = json.load(f)
    WT, TC, ET, names = [], [], [], []
    for case_id, case in data.items():
        WT.append(case['WT'][0])
        TC.append(case['TC'][0])
        ET.append(case['ET'][0])
        names.append(case_id)
    return pd.DataFrame(zip(WT, TC, ET), columns=['WT dice', 'TC dice', 'ET dice'], index=names)


def get_overlaps(unet_df, TransBTS_df, nnunet_df,
                 WT_thresh=0.91, TC_thresh=0.86, ET_thresh=0.85):
    def bad_idx(df):
        return df[(df['WT dice'] < WT_thresh) &
                  (df['TC dice'] < TC_thresh) &
                  (df['ET dice'] < ET_thresh)].index.tolist()
    u = set(bad_idx(unet_df))
    n = set(bad_idx(nnunet_df))
    t = set(bad_idx(TransBTS_df))
    return list(u & n & t), list(u & n)


def read_radiomics(analysis_type, location):
    path = f'../../Results/Analysis_Results/Radiomics/{location}/{analysis_type}.pkl'
    with open(path, 'rb') as f:
        return pkl.load(f)


def load_texture(analysis_type, location, performance_df):
    """Load one texture family for one subregion, merge with performance_df, dropna."""
    raw = read_radiomics(analysis_type, location)
    frames = []
    for prop_name, prop_data in raw.items():
        pf = pd.DataFrame.from_dict(prop_data, orient='index').astype(float)
        pf.columns = [f'{prop_name}_{c}_{analysis_type}' for c in pf.columns]
        frames.append(pf)
    feat_df = pd.concat(frames, axis=1)
    merged = feat_df.merge(performance_df, left_index=True, right_index=True)
    merged = merged.T.drop_duplicates().T.dropna()
    return merged


def split_good_bad(df, unet_nnunet_overlaps,
                   WT_thresh=0.91, TC_thresh=0.86, ET_thresh=0.85):
    bad_mask = ((df['WT dice'] < WT_thresh) &
                (df['TC dice'] < TC_thresh) &
                (df['ET dice'] < ET_thresh))
    good_mask = ((df['WT dice'] >= WT_thresh) &
                 (df['TC dice'] >= TC_thresh) &
                 (df['ET dice'] >= ET_thresh))
    bad  = df[bad_mask & df.index.isin(unet_nnunet_overlaps)].drop(
               ['WT dice', 'TC dice', 'ET dice'], axis=1)
    good = df[good_mask].drop(['WT dice', 'TC dice', 'ET dice'], axis=1)
    return (good.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all'),
            bad.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all'))

## Load Performance Data

In [3]:
WT_THRESH, TC_THRESH, ET_THRESH = 0.91, 0.86, 0.85

unet_df     = load_unet_result('../../Results/Result/Vanilla_Unet/Unet_test_dice.csv')
nnunet_df   = load_nnunet_result('../../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json')
transbts_df = load_TransBTS_result('../../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json')

_, unet_nnunet_overlaps = get_overlaps(unet_df, transbts_df, nnunet_df)
performance_df = unet_df.copy()

print(f'Total cases:           {len(unet_df)}')
print(f'Concordant poor:       {len(unet_nnunet_overlaps)}')
good_n = ((unet_df['WT dice'] >= WT_THRESH) &
          (unet_df['TC dice'] >= TC_THRESH) &
          (unet_df['ET dice'] >= ET_THRESH)).sum()
print(f'Concordant good:       {good_n}')

Total cases:           939
Concordant poor:       47
Concordant good:       404


## Plotting Helpers

In [4]:
def cliffs_delta(x, y):
    nx, ny = len(x), len(y)
    n_gr = sum(xi > yi for xi in x for yi in y)
    n_ls = sum(xi < yi for xi in x for yi in y)
    return (n_gr - n_ls) / (nx * ny)


def categorize_effect(delta):
    a = abs(delta)
    if a < 0.147:  return 'No'
    if a < 0.33:   return 'Small'
    if a < 0.474:  return 'Medium'
    return 'Large'


def add_sig(ax, good, bad, x_center, alpha=0.05):
    """Annotate significance + effect size above a subregion group."""
    _, p = mannwhitneyu(good.dropna(), bad.dropna())
    if p < alpha:
        d = cliffs_delta(good.dropna().tolist(), bad.dropna().tolist())
        eff = categorize_effect(d)
        ax2 = ax.secondary_xaxis('top')
        ax2.set_xticks([x_center])
        ax2.set_xticklabels([f'* {eff}'], fontsize=8, color='blue')
        return p, d, eff
    return p, None, 'No'


COLORS        = ['lightblue', 'lightcoral'] * 3   # Good/Bad alternating for WT, TC, ET
MODALITIES    = ['flair', 't2', 't1', 't1ce']
MOD_LABELS    = ['FLAIR', 'T2', 'T1', 'T1CE']
SR_LABELS     = ['WT', 'TC', 'ET']
FEATS_PER_PAGE = 4


def plot_texture_subregion(family_key, feature_names, col_prefix, analysis_type,
                           wt_good, wt_bad, tc_good, tc_bad, et_good, et_bad,
                           out_pdf_prefix, summary_dict):
    """
    Layout: rows = up to FEATS_PER_PAGE features, cols = 4 modalities.
    Each group of 4 features is saved as a separate PDF:
      out_pdf_prefix_1.pdf, out_pdf_prefix_2.pdf, ...
    """
    if family_key not in summary_dict:
        summary_dict[family_key] = {}
    family_summary = summary_dict[family_key]

    # chunk features into pages of FEATS_PER_PAGE
    pages = [feature_names[i:i+FEATS_PER_PAGE]
             for i in range(0, len(feature_names), FEATS_PER_PAGE)]

    saved_files = []
    for page_num, page_feats in enumerate(pages, start=1):
        out_pdf = f'{out_pdf_prefix}_{page_num}.pdf'
        n_rows = len(page_feats)
        fig, axes = plt.subplots(n_rows, 4,
                                 figsize=(20, 5 * n_rows),
                                 squeeze=False)

        # Column headers (modalities) on the top row
        for col_idx, mod_label in enumerate(MOD_LABELS):
            axes[0, col_idx].set_title(mod_label, fontsize=12, fontweight='bold', pad=12)

        for row_idx, feat in enumerate(page_feats):
            if feat not in family_summary:
                family_summary[feat] = {}

            for col_idx, (mod, mod_label) in enumerate(zip(MODALITIES, MOD_LABELS)):
                ax = axes[row_idx, col_idx]
                col = f'{col_prefix}{feat}_{mod}_{analysis_type}'

                if mod_label not in family_summary[feat]:
                    family_summary[feat][mod_label] = {}

                groups = [
                    (wt_good, wt_bad, 'WT'),
                    (tc_good, tc_bad, 'TC'),
                    (et_good, et_bad, 'ET'),
                ]

                plot_data = []
                for gd, bd, _ in groups:
                    g = gd[col].dropna() if col in gd.columns else pd.Series(dtype=float)
                    b = bd[col].dropna() if col in bd.columns else pd.Series(dtype=float)
                    plot_data += [g, b]

                box = ax.boxplot(plot_data, patch_artist=True, widths=0.5, showfliers=False)
                for patch, color in zip(box['boxes'], COLORS):
                    patch.set_facecolor(color)
                for med in box['medians']:
                    med.set_color('black')

                # Dashed separators between subregion groups
                for sep in [2.5, 4.5]:
                    ax.axvline(sep, color='grey', linestyle='--', linewidth=0.7, alpha=0.6)

                # Significance annotations and JSON summary
                for sr_idx, (gd, bd, sr) in enumerate(groups):
                    g = gd[col].dropna() if col in gd.columns else pd.Series(dtype=float)
                    b = bd[col].dropna() if col in bd.columns else pd.Series(dtype=float)
                    x_center = 2 * sr_idx + 1.5   # centre of the good/bad pair
                    p, d, eff = add_sig(ax, g, b, x_center)
                    family_summary[feat][mod_label][sr] = {
                        'median_good': float(g.median()) if len(g) else None,
                        'median_bad':  float(b.median()) if len(b) else None,
                        'p_value':     float(p),
                        'cliffs_delta': float(d) if d is not None else None,
                        'effect_size': eff,
                    }

                # x-tick labels: WT / TC / ET
                ax.set_xticks([1.5, 3.5, 5.5])
                ax.set_xticklabels(SR_LABELS, fontsize=10)

                # Feature label on left y-axis of first column only
                if col_idx == 0:
                    ax.set_ylabel(feat, fontsize=10, labelpad=6)

        plt.tight_layout(rect=[0, 0, 1, 0.97])
        fig.suptitle(f'{family_key.upper()} — Subregion Analysis (Part {page_num})',
                     fontsize=14, fontweight='bold', y=1.0)
        fig.savefig(out_pdf, bbox_inches='tight')
        plt.close(fig)
        saved_files.append(out_pdf)
        print(f'  Saved: {out_pdf}')

    return saved_files


## Load Texture Data for All Three Subregions

In [5]:
# --- GLCM ---
print('Loading GLCM...')
wt_glcm = load_texture('glcm_10', 'Tumor_WT', performance_df)
tc_glcm = load_texture('glcm_10', 'Tumor_TC', performance_df)
et_glcm = load_texture('glcm_10', 'Tumor_ET', performance_df)
wt_glcm_good, wt_glcm_bad = split_good_bad(wt_glcm, unet_nnunet_overlaps)
tc_glcm_good, tc_glcm_bad = split_good_bad(tc_glcm, unet_nnunet_overlaps)
et_glcm_good, et_glcm_bad = split_good_bad(et_glcm, unet_nnunet_overlaps)
print(f'  WT  good={len(wt_glcm_good)}, bad={len(wt_glcm_bad)}')
print(f'  TC  good={len(tc_glcm_good)}, bad={len(tc_glcm_bad)}')
print(f'  ET  good={len(et_glcm_good)}, bad={len(et_glcm_bad)}')

# --- GLDM ---
print('Loading GLDM...')
wt_gldm = load_texture('gldm_10', 'Tumor_WT', performance_df)
tc_gldm = load_texture('gldm_10', 'Tumor_TC', performance_df)
et_gldm = load_texture('gldm_10', 'Tumor_ET', performance_df)
wt_gldm_good, wt_gldm_bad = split_good_bad(wt_gldm, unet_nnunet_overlaps)
tc_gldm_good, tc_gldm_bad = split_good_bad(tc_gldm, unet_nnunet_overlaps)
et_gldm_good, et_gldm_bad = split_good_bad(et_gldm, unet_nnunet_overlaps)
print(f'  WT  good={len(wt_gldm_good)}, bad={len(wt_gldm_bad)}')

# --- GLRLM ---
print('Loading GLRLM...')
wt_glrlm = load_texture('glrlm', 'Tumor_WT', performance_df)
tc_glrlm = load_texture('glrlm', 'Tumor_TC', performance_df)
et_glrlm = load_texture('glrlm', 'Tumor_ET', performance_df)
wt_glrlm_good, wt_glrlm_bad = split_good_bad(wt_glrlm, unet_nnunet_overlaps)
tc_glrlm_good, tc_glrlm_bad = split_good_bad(tc_glrlm, unet_nnunet_overlaps)
et_glrlm_good, et_glrlm_bad = split_good_bad(et_glrlm, unet_nnunet_overlaps)
print(f'  WT  good={len(wt_glrlm_good)}, bad={len(wt_glrlm_bad)}')

# --- GLSZM ---
print('Loading GLSZM...')
wt_glszm = load_texture('glszm', 'Tumor_WT', performance_df)
tc_glszm = load_texture('glszm', 'Tumor_TC', performance_df)
et_glszm = load_texture('glszm', 'Tumor_ET', performance_df)
wt_glszm_good, wt_glszm_bad = split_good_bad(wt_glszm, unet_nnunet_overlaps)
tc_glszm_good, tc_glszm_bad = split_good_bad(tc_glszm, unet_nnunet_overlaps)
et_glszm_good, et_glszm_bad = split_good_bad(et_glszm, unet_nnunet_overlaps)
print(f'  WT  good={len(wt_glszm_good)}, bad={len(wt_glszm_bad)}')

# --- NGTDM ---
print('Loading NGTDM...')
wt_ngtdm = load_texture('ngtdm_10', 'Tumor_WT', performance_df)
tc_ngtdm = load_texture('ngtdm_10', 'Tumor_TC', performance_df)
et_ngtdm = load_texture('ngtdm_10', 'Tumor_ET', performance_df)
wt_ngtdm_good, wt_ngtdm_bad = split_good_bad(wt_ngtdm, unet_nnunet_overlaps)
tc_ngtdm_good, tc_ngtdm_bad = split_good_bad(tc_ngtdm, unet_nnunet_overlaps)
et_ngtdm_good, et_ngtdm_bad = split_good_bad(et_ngtdm, unet_nnunet_overlaps)
print(f'  WT  good={len(wt_ngtdm_good)}, bad={len(wt_ngtdm_bad)}')

Loading GLCM...
  WT  good=404, bad=47
  TC  good=404, bad=45
  ET  good=404, bad=45
Loading GLDM...
  WT  good=404, bad=47
Loading GLRLM...
  WT  good=404, bad=47
Loading GLSZM...
  WT  good=404, bad=47
Loading NGTDM...
  WT  good=404, bad=47


## Feature Name Definitions

In [6]:
GLCM_FEATURES = [
    'Autocorrelation', 'ClusterProminence', 'ClusterShade', 'ClusterTendency',
    'Contrast', 'Correlation', 'DifferenceAverage', 'DifferenceEntropy',
    'DifferenceVariance', 'Id', 'Idm', 'Idmn', 'Idn', 'Imc1', 'Imc2',
    'InverseVariance', 'JointAverage', 'JointEnergy', 'JointEntropy',
    'MaximumProbability', 'MCC', 'SumAverage', 'SumEntropy', 'SumSquares',
]

GLDM_FEATURES = [
    'DependenceEntropy', 'DependenceNonUniformity', 'DependenceNonUniformityNormalized',
    'DependenceVariance', 'GrayLevelNonUniformity', 'GrayLevelVariance',
    'HighGrayLevelEmphasis', 'LargeDependenceEmphasis',
    'LargeDependenceHighGrayLevelEmphasis', 'LargeDependenceLowGrayLevelEmphasis',
    'LowGrayLevelEmphasis', 'SmallDependenceEmphasis',
    'SmallDependenceHighGrayLevelEmphasis', 'SmallDependenceLowGrayLevelEmphasis',
]

GLRLM_FEATURES = [
    'GrayLevelNonUniformity', 'GrayLevelNonUniformityNormalized', 'GrayLevelVariance',
    'HighGrayLevelRunEmphasis', 'LongRunEmphasis',
    'LongRunHighGrayLevelEmphasis', 'LongRunLowGrayLevelEmphasis',
    'LowGrayLevelRunEmphasis', 'RunEntropy', 'RunLengthNonUniformity',
    'RunLengthNonUniformityNormalized', 'RunPercentage', 'RunVariance',
    'ShortRunEmphasis', 'ShortRunHighGrayLevelEmphasis', 'ShortRunLowGrayLevelEmphasis',
]

GLSZM_FEATURES = [
    'GrayLevelNonUniformity', 'GrayLevelNonUniformityNormalized', 'GrayLevelVariance',
    'HighGrayLevelZoneEmphasis', 'LargeAreaEmphasis',
    'LargeAreaHighGrayLevelEmphasis', 'LargeAreaLowGrayLevelEmphasis',
    'LowGrayLevelZoneEmphasis', 'SizeZoneNonUniformity',
    'SizeZoneNonUniformityNormalized', 'SmallAreaEmphasis',
    'SmallAreaHighGrayLevelEmphasis', 'SmallAreaLowGrayLevelEmphasis',
    'ZoneEntropy', 'ZonePercentage', 'ZoneVariance',
]

NGTDM_FEATURES = ['Busyness', 'Coarseness', 'Complexity', 'Contrast', 'Strength']

## Generate Charts — GLCM

In [7]:
summary = {}

plot_texture_subregion(
    family_key    = 'GLCM',
    feature_names = GLCM_FEATURES,
    col_prefix    = 'original_glcm_',
    analysis_type = 'glcm_10',
    wt_good=wt_glcm_good, wt_bad=wt_glcm_bad,
    tc_good=tc_glcm_good, tc_bad=tc_glcm_bad,
    et_good=et_glcm_good, et_bad=et_glcm_bad,
    out_pdf_prefix = '../../Results/final_figures/texture_subregion_glcm',
    summary_dict  = summary,
)

  Saved: ../../Results/final_figures/texture_subregion_glcm_1.pdf
  Saved: ../../Results/final_figures/texture_subregion_glcm_2.pdf
  Saved: ../../Results/final_figures/texture_subregion_glcm_3.pdf
  Saved: ../../Results/final_figures/texture_subregion_glcm_4.pdf
  Saved: ../../Results/final_figures/texture_subregion_glcm_5.pdf
  Saved: ../../Results/final_figures/texture_subregion_glcm_6.pdf


['../../Results/final_figures/texture_subregion_glcm_1.pdf',
 '../../Results/final_figures/texture_subregion_glcm_2.pdf',
 '../../Results/final_figures/texture_subregion_glcm_3.pdf',
 '../../Results/final_figures/texture_subregion_glcm_4.pdf',
 '../../Results/final_figures/texture_subregion_glcm_5.pdf',
 '../../Results/final_figures/texture_subregion_glcm_6.pdf']

## Generate Charts — GLDM

In [8]:
plot_texture_subregion(
    family_key    = 'GLDM',
    feature_names = GLDM_FEATURES,
    col_prefix    = 'original_gldm_',
    analysis_type = 'gldm_10',
    wt_good=wt_gldm_good, wt_bad=wt_gldm_bad,
    tc_good=tc_gldm_good, tc_bad=tc_gldm_bad,
    et_good=et_gldm_good, et_bad=et_gldm_bad,
    out_pdf_prefix = '../../Results/final_figures/texture_subregion_gldm',
    summary_dict  = summary,
)

  Saved: ../../Results/final_figures/texture_subregion_gldm_1.pdf
  Saved: ../../Results/final_figures/texture_subregion_gldm_2.pdf
  Saved: ../../Results/final_figures/texture_subregion_gldm_3.pdf
  Saved: ../../Results/final_figures/texture_subregion_gldm_4.pdf


['../../Results/final_figures/texture_subregion_gldm_1.pdf',
 '../../Results/final_figures/texture_subregion_gldm_2.pdf',
 '../../Results/final_figures/texture_subregion_gldm_3.pdf',
 '../../Results/final_figures/texture_subregion_gldm_4.pdf']

## Generate Charts — GLRLM

In [9]:
plot_texture_subregion(
    family_key    = 'GLRLM',
    feature_names = GLRLM_FEATURES,
    col_prefix    = 'original_glrlm_',
    analysis_type = 'glrlm',
    wt_good=wt_glrlm_good, wt_bad=wt_glrlm_bad,
    tc_good=tc_glrlm_good, tc_bad=tc_glrlm_bad,
    et_good=et_glrlm_good, et_bad=et_glrlm_bad,
    out_pdf_prefix = '../../Results/final_figures/texture_subregion_glrlm',
    summary_dict  = summary,
)

  Saved: ../../Results/final_figures/texture_subregion_glrlm_1.pdf
  Saved: ../../Results/final_figures/texture_subregion_glrlm_2.pdf
  Saved: ../../Results/final_figures/texture_subregion_glrlm_3.pdf
  Saved: ../../Results/final_figures/texture_subregion_glrlm_4.pdf


['../../Results/final_figures/texture_subregion_glrlm_1.pdf',
 '../../Results/final_figures/texture_subregion_glrlm_2.pdf',
 '../../Results/final_figures/texture_subregion_glrlm_3.pdf',
 '../../Results/final_figures/texture_subregion_glrlm_4.pdf']

## Generate Charts — GLSZM

In [10]:
plot_texture_subregion(
    family_key    = 'GLSZM',
    feature_names = GLSZM_FEATURES,
    col_prefix    = 'original_glszm_',
    analysis_type = 'glszm',
    wt_good=wt_glszm_good, wt_bad=wt_glszm_bad,
    tc_good=tc_glszm_good, tc_bad=tc_glszm_bad,
    et_good=et_glszm_good, et_bad=et_glszm_bad,
    out_pdf_prefix = '../../Results/final_figures/texture_subregion_glszm',
    summary_dict  = summary,
)

  Saved: ../../Results/final_figures/texture_subregion_glszm_1.pdf
  Saved: ../../Results/final_figures/texture_subregion_glszm_2.pdf
  Saved: ../../Results/final_figures/texture_subregion_glszm_3.pdf
  Saved: ../../Results/final_figures/texture_subregion_glszm_4.pdf


['../../Results/final_figures/texture_subregion_glszm_1.pdf',
 '../../Results/final_figures/texture_subregion_glszm_2.pdf',
 '../../Results/final_figures/texture_subregion_glszm_3.pdf',
 '../../Results/final_figures/texture_subregion_glszm_4.pdf']

## Generate Charts — NGTDM

In [11]:
plot_texture_subregion(
    family_key    = 'NGTDM',
    feature_names = NGTDM_FEATURES,
    col_prefix    = 'original_ngtdm_',
    analysis_type = 'ngtdm_10',
    wt_good=wt_ngtdm_good, wt_bad=wt_ngtdm_bad,
    tc_good=tc_ngtdm_good, tc_bad=tc_ngtdm_bad,
    et_good=et_ngtdm_good, et_bad=et_ngtdm_bad,
    out_pdf_prefix = '../../Results/final_figures/texture_subregion_ngtdm',
    summary_dict  = summary,
)

  Saved: ../../Results/final_figures/texture_subregion_ngtdm_1.pdf
  Saved: ../../Results/final_figures/texture_subregion_ngtdm_2.pdf


['../../Results/final_figures/texture_subregion_ngtdm_1.pdf',
 '../../Results/final_figures/texture_subregion_ngtdm_2.pdf']

## Save JSON Summary

In [12]:
out_json = '../../Results/Json_summary/summary_texture_subregion.json'
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=4)
print(f'JSON saved: {out_json}')

JSON saved: ../../Results/Json_summary/summary_texture_subregion.json


## Summary Table — Significant Features per Family × Subregion

In [13]:
rows = []
for family, feats in summary.items():
    for feat, mods in feats.items():
        for mod, srs in mods.items():
            for sr, stats in srs.items():
                rows.append({
                    'Family':       family,
                    'Feature':      feat,
                    'Modality':     mod,
                    'Subregion':    sr,
                    'Median Good':  round(stats['median_good'], 4) if stats['median_good'] else None,
                    'Median Bad':   round(stats['median_bad'],  4) if stats['median_bad']  else None,
                    'p-value':      f"{stats['p_value']:.3e}",
                    "Cliff's Delta": round(stats['cliffs_delta'], 3) if stats['cliffs_delta'] else 'n.s.',
                    'Effect Size':  stats['effect_size'],
                })

results_df = pd.DataFrame(rows)
sig_df = results_df[results_df['Effect Size'] != 'No']

print(f'Total comparisons:    {len(results_df)}')
print(f'Significant (p<0.05): {len(sig_df)}')
print(f'  Small:  {(sig_df["Effect Size"]=="Small").sum()}')
print(f'  Medium: {(sig_df["Effect Size"]=="Medium").sum()}')
print(f'  Large:  {(sig_df["Effect Size"]=="Large").sum()}')
print()

print('=== Significant counts by Family × Subregion ===')
print(sig_df.groupby(['Family', 'Subregion']).size().unstack(fill_value=0))

print('\n=== Large effects only ===')
large_df = sig_df[sig_df['Effect Size'] == 'Large']
print(large_df[['Family','Feature','Modality','Subregion','Median Good','Median Bad',"Cliff's Delta"]].to_string(index=False))

Total comparisons:    900
Significant (p<0.05): 539
  Small:  210
  Medium: 161
  Large:  168

=== Significant counts by Family × Subregion ===
Subregion  ET  TC  WT
Family               
GLCM       54  56  57
GLDM       47  47  42
GLRLM      25  29  42
GLSZM      26  31  37
NGTDM      14  15  17

=== Large effects only ===
Family                              Feature Modality Subregion   Median Good  Median Bad Cliff's Delta
  GLCM                          Correlation    FLAIR        TC  6.200000e-02     -0.0791         0.527
  GLCM                          Correlation    FLAIR        ET  7.870000e-02     -0.0562         0.594
  GLCM                          Correlation       T2        TC  6.460000e-02     -0.0650         0.565
  GLCM                          Correlation       T2        ET  6.850000e-02     -0.0443         0.649
  GLCM                          Correlation       T1        TC  1.058000e-01     -0.0692         0.526
  GLCM                          Correlation       T1    

## Consistency Check — Which WT findings replicate in TC and ET?

In [14]:
for family in summary.keys():
    fam_df = sig_df[sig_df['Family'] == family]
    wt_sig = set(zip(fam_df[fam_df['Subregion']=='WT']['Feature'],
                     fam_df[fam_df['Subregion']=='WT']['Modality']))
    tc_sig = set(zip(fam_df[fam_df['Subregion']=='TC']['Feature'],
                     fam_df[fam_df['Subregion']=='TC']['Modality']))
    et_sig = set(zip(fam_df[fam_df['Subregion']=='ET']['Feature'],
                     fam_df[fam_df['Subregion']=='ET']['Modality']))

    print(f'--- {family} ---')
    print(f'  WT significant: {len(wt_sig)}')
    print(f'  TC significant: {len(tc_sig)}  (also in WT: {len(wt_sig & tc_sig)})')
    print(f'  ET significant: {len(et_sig)}  (also in WT: {len(wt_sig & et_sig)})')
    tc_only = tc_sig - wt_sig
    et_only = et_sig - wt_sig
    if tc_only:
        print(f'  TC-only (not in WT): {sorted(tc_only)}')
    if et_only:
        print(f'  ET-only (not in WT): {sorted(et_only)}')
    print()

--- GLCM ---
  WT significant: 57
  TC significant: 56  (also in WT: 42)
  ET significant: 54  (also in WT: 41)
  TC-only (not in WT): [('Correlation', 'FLAIR'), ('Correlation', 'T1'), ('Correlation', 'T2'), ('DifferenceEntropy', 'T1CE'), ('Idmn', 'T1'), ('Idn', 'T1'), ('JointEnergy', 'FLAIR'), ('JointEnergy', 'T1CE'), ('JointEntropy', 'FLAIR'), ('JointEntropy', 'T1CE'), ('MaximumProbability', 'FLAIR'), ('MaximumProbability', 'T1CE'), ('SumEntropy', 'FLAIR'), ('SumEntropy', 'T1CE')]
  ET-only (not in WT): [('Correlation', 'FLAIR'), ('Correlation', 'T1'), ('Correlation', 'T2'), ('Idmn', 'T1'), ('Idn', 'T1'), ('JointEnergy', 'FLAIR'), ('JointEnergy', 'T1CE'), ('JointEntropy', 'FLAIR'), ('JointEntropy', 'T1CE'), ('MaximumProbability', 'FLAIR'), ('MaximumProbability', 'T1CE'), ('SumEntropy', 'FLAIR'), ('SumEntropy', 'T1CE')]

--- GLDM ---
  WT significant: 42
  TC significant: 47  (also in WT: 37)
  ET significant: 47  (also in WT: 37)
  TC-only (not in WT): [('DependenceNonUniformity', 'T